In [1]:
import random
import json
import numpy as np

# Round down to nearest divisor
def round_down(num, divisor):
    return num - (num%divisor)

# Custom JSON encoder to maintain integer keys
class IntKeyDictEncoder(json.JSONEncoder):
    def encode(self, obj):
        if isinstance(obj, dict):
            return '{' + ', '.join(f'{k}: {self.encode(v)}' for k, v in obj.items()) + '}'
        return json.JSONEncoder.encode(self, obj)
    
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines
    inv_slack = 1-slack

    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)
        release = random.choices([0, release_], weights=[slack, inv_slack])[0]
        # deadline_ = min(release + duration + random.randint(10, 50), max_makespan)
        deadline_ = release + duration
        deadline = random.choices([max_makespan, deadline_], weights=[slack, inv_slack])[0]
        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)
        release = random.choices([0, release_], weights=[slack, inv_slack])[0]
        # deadline_ = min(release + duration + random.randint(10, 50), max_makespan)
        deadline_ = release + duration
        deadline = random.choices([max_makespan, deadline_], weights=[slack, inv_slack])[0]
        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = total_duration_0  # Assuming the makespans are already balanced
    return jobs, [makespan, makespan]

def calculate_slack(data):
    # Determine the minimum release value and maximum deadline value in the dictionary
    min_release = min(item['release'] for item in data.values())
    max_deadline = max(item['deadline'] for item in data.values())

    # Total number of entries in the dictionary
    total_entries = len(data)
    
    # Count the number of entries with release == min_release and deadline == max_deadline
    count_matching_entries = sum(
        1 for item in data.values() if item['release'] == min_release and item['deadline'] == max_deadline
    )
    
    # Calculate the percentage
    percentage = (count_matching_entries / total_entries) * 100
    
    print(f'Problem instance has {percentage}% slackness')

In [40]:
def generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Calculate the number of jobs that should have maximum slack
    slack_jobs_count = int(slack * num_jobs)
    non_slack_jobs_count = num_jobs - slack_jobs_count
    
    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)

        # Determine if this job should contribute to slack or not
        if i < non_slack_jobs_count // 2:
            # Normal job, doesn't contribute to slack
            release = release_
            deadline_ = release + duration
            deadline = min(int(deadline_ * 2), max_makespan)
        else:
            # Slack job
            release = 0
            deadline = max_makespan

        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)

        # Determine if this job should contribute to slack or not
        if i < non_slack_jobs_count // 2:
            # Normal job, doesn't contribute to slack
            release = release_
            deadline_ = release + duration
            deadline = min(int(deadline_ * 2), max_makespan)
        else:
            # Slack job
            release = 0
            deadline = max_makespan

        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = max(total_duration_0, total_duration_1)
    return jobs, [makespan, makespan]

In [41]:
# under 5% overlap of the total possible combinations

# ([19, 1], [18, 2]) #4 jobs
# ([20, 18, 1, 1], [19, 15, 3, 3]) #8 jobs
# ([20, 20, 20, 16, 1, 1, 1, 1], [19, 19, 19, 11, 4, 3, 3, 2]) #16 jobs
# ([20, 20, 20, 20, 6, 1, 1, 1, 1], [19, 19, 19, 17, 7, 3, 2, 2, 2]) #18 jobs
# ([20, 20, 20, 20, 15, 1, 1, 1, 1, 1], [19, 19, 19, 19, 10, 4, 3, 3, 2, 2]) #20 jobs
# ([20, 20, 20, 20, 20, 14, 1, 1, 1, 1, 1, 1], [19, 19, 19, 19, 19, 9, 4, 4, 2, 2, 2, 2]) #22 jobs

In [42]:
num_machines = 2
duration_set_0 = [121,
  79,
  156,
  219,
  243,
  63,
  186,
  161,
  138,
  170,
  7,
  5,
  49,
  110,
  22,
  234,
  154,
  245,
  76,
  26,
  114,
  244,
  246,
  160,
  144,
  186,
  161,
  226,
  112,
  237,
  49,
  68,
  175,
  178,
  132,
  250,
  128,
  120,
  191,
  222,
  118,
  16,
  10,
  231,
  135,
  203,
  44,
  228,
  4,
  14]
duration_set_1 =  [168,
  141,
  78,
  1,
  180,
  20,
  51,
  190,
  60,
  100,
  137,
  149,
  55,
  124,
  104,
  177,
  177,
  50,
  195,
  238,
  163,
  234,
  101,
  36,
  244,
  106,
  180,
  196,
  162,
  107,
  149,
  71,
  184,
  233,
  227,
  172,
  252,
  44,
  41,
  187,
  84,
  247,
  115,
  36,
  184,
  191,
  112,
  64,
  213,
  80]

num_jobs = len(duration_set_0) + len(duration_set_1)
slack = 0.5
slack_str = int(slack*100)
json_path = f'../data_gecco/ssjsp/ssjsp_{num_jobs}_s{slack_str}.json'
max_makespan = sum(duration_set_0)

jobs, makespans = generate_jobs_from_duration_sets(num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack)

In [43]:
num_jobs, len(duration_set_0), len(duration_set_1)

(100, 50, 50)

In [44]:
jobs, makespans

({1: {'duration': 121, 'release': 0, 'deadline': 242, 'machine': 0},
  2: {'duration': 79, 'release': 120, 'deadline': 398, 'machine': 0},
  3: {'duration': 156, 'release': 200, 'deadline': 712, 'machine': 0},
  4: {'duration': 219, 'release': 350, 'deadline': 1138, 'machine': 0},
  5: {'duration': 243, 'release': 570, 'deadline': 1626, 'machine': 0},
  6: {'duration': 63, 'release': 810, 'deadline': 1746, 'machine': 0},
  7: {'duration': 186, 'release': 880, 'deadline': 2132, 'machine': 0},
  8: {'duration': 161, 'release': 1060, 'deadline': 2442, 'machine': 0},
  9: {'duration': 138, 'release': 1220, 'deadline': 2716, 'machine': 0},
  10: {'duration': 170, 'release': 1360, 'deadline': 3060, 'machine': 0},
  11: {'duration': 7, 'release': 1530, 'deadline': 3074, 'machine': 0},
  12: {'duration': 5, 'release': 1540, 'deadline': 3090, 'machine': 0},
  13: {'duration': 49, 'release': 1540, 'deadline': 3178, 'machine': 0},
  14: {'duration': 110, 'release': 1590, 'deadline': 3400, 'machin

In [45]:
calculate_slack(jobs)

Problem instance has 50.0% slackness


In [46]:
# Save the generated jobs to a JSON file
with open(json_path, 'w') as f:
    json.dump(jobs, f, indent=4, cls=IntKeyDictEncoder)

In [22]:
filename = 'ssjsp_100_s10'
json_path = f'../data_gecco/ssjsp/{filename}.json'

In [23]:
with open(json_path) as f:
    d = json.load(f)
    print(d)

{'1': {'duration': 121, 'release': 0, 'deadline': 169, 'machine': 0}, '2': {'duration': 79, 'release': 120, 'deadline': 278, 'machine': 0}, '3': {'duration': 156, 'release': 200, 'deadline': 498, 'machine': 0}, '4': {'duration': 219, 'release': 350, 'deadline': 796, 'machine': 0}, '5': {'duration': 243, 'release': 570, 'deadline': 1138, 'machine': 0}, '6': {'duration': 63, 'release': 810, 'deadline': 1222, 'machine': 0}, '7': {'duration': 186, 'release': 880, 'deadline': 1492, 'machine': 0}, '8': {'duration': 161, 'release': 1060, 'deadline': 1709, 'machine': 0}, '9': {'duration': 138, 'release': 1220, 'deadline': 1901, 'machine': 0}, '10': {'duration': 170, 'release': 1360, 'deadline': 2142, 'machine': 0}, '11': {'duration': 7, 'release': 1530, 'deadline': 2151, 'machine': 0}, '12': {'duration': 5, 'release': 1540, 'deadline': 2163, 'machine': 0}, '13': {'duration': 49, 'release': 1540, 'deadline': 2224, 'machine': 0}, '14': {'duration': 110, 'release': 1590, 'deadline': 2380, 'machin

In [24]:
d

{'1': {'duration': 121, 'release': 0, 'deadline': 169, 'machine': 0},
 '2': {'duration': 79, 'release': 120, 'deadline': 278, 'machine': 0},
 '3': {'duration': 156, 'release': 200, 'deadline': 498, 'machine': 0},
 '4': {'duration': 219, 'release': 350, 'deadline': 796, 'machine': 0},
 '5': {'duration': 243, 'release': 570, 'deadline': 1138, 'machine': 0},
 '6': {'duration': 63, 'release': 810, 'deadline': 1222, 'machine': 0},
 '7': {'duration': 186, 'release': 880, 'deadline': 1492, 'machine': 0},
 '8': {'duration': 161, 'release': 1060, 'deadline': 1709, 'machine': 0},
 '9': {'duration': 138, 'release': 1220, 'deadline': 1901, 'machine': 0},
 '10': {'duration': 170, 'release': 1360, 'deadline': 2142, 'machine': 0},
 '11': {'duration': 7, 'release': 1530, 'deadline': 2151, 'machine': 0},
 '12': {'duration': 5, 'release': 1540, 'deadline': 2163, 'machine': 0},
 '13': {'duration': 49, 'release': 1540, 'deadline': 2224, 'machine': 0},
 '14': {'duration': 110, 'release': 1590, 'deadline': 

In [26]:
machine_0_durations = [job['duration'] for job in d.values() if job['machine'] == 0]
machine_1_durations = [job['duration'] for job in d.values() if job['machine'] == 1]

In [27]:
machine_0_durations, machine_1_durations

([121,
  79,
  156,
  219,
  243,
  63,
  186,
  161,
  138,
  170,
  7,
  5,
  49,
  110,
  22,
  234,
  154,
  245,
  76,
  26,
  114,
  244,
  246,
  160,
  144,
  186,
  161,
  226,
  112,
  237,
  49,
  68,
  175,
  178,
  132,
  250,
  128,
  120,
  191,
  222,
  118,
  16,
  10,
  231,
  135,
  203,
  44,
  228,
  4,
  14],
 [168,
  141,
  78,
  1,
  180,
  20,
  51,
  190,
  60,
  100,
  137,
  149,
  55,
  124,
  104,
  177,
  177,
  50,
  195,
  238,
  163,
  234,
  101,
  36,
  244,
  106,
  180,
  196,
  162,
  107,
  149,
  71,
  184,
  233,
  227,
  172,
  252,
  44,
  41,
  187,
  84,
  247,
  115,
  36,
  184,
  191,
  112,
  64,
  213,
  80])

In [38]:
import numpy as np

def generate_jobs_from_duration_sets(
    num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, resource_tightness
):
    jobs = {}
    job_count_per_machine = num_jobs // num_machines

    # Calculate the number of jobs that should have maximum slack
    slack_jobs_count = int(slack * num_jobs)
    non_slack_jobs_count = num_jobs - slack_jobs_count

    # Create jobs for machine 0
    total_duration_0 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_0[i]
        release_ = round_down(total_duration_0, 10)

        if i < non_slack_jobs_count // 2:
            release = release_
            deadline_ = release + duration
            deadline = min(int(deadline_ * 1.4), max_makespan)
        else:
            release = 0
            deadline = max_makespan

        jobs[i + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 0}
        total_duration_0 += duration

    # Create jobs for machine 1
    total_duration_1 = 0
    for i in range(job_count_per_machine):
        duration = duration_set_1[i]
        release_ = round_down(total_duration_1, 10)

        if i < non_slack_jobs_count // 2:
            release = release_
            deadline_ = release + duration
            deadline = min(int(deadline_ * 1.4), max_makespan)
        else:
            release = 0
            deadline = max_makespan

        jobs[i + job_count_per_machine + 1] = {"duration": duration, "release": release, "deadline": deadline, "machine": 1}
        total_duration_1 += duration

    makespan = max(total_duration_0, total_duration_1)

    # Introduce resource constraints
    for job_id, job in jobs.items():
        job["resources"] = np.random.randint(1, resource_tightness + 1)  # Adjust based on tightness

    return jobs, [makespan, makespan]

def compute_sigma_distance(jobs, max_makespan):
    """
    Compute the sigma distance indicator for a given job set.
    """
    # Placeholder for sampling or enumerating solutions
    sampled_makespans = [np.random.randint(1, max_makespan) for _ in range(100)]

    avg_makespan = np.mean(sampled_makespans)
    std_dev = np.std(sampled_makespans)
    reference_makespan = min(sampled_makespans)  # Assuming a heuristic for near-optimal

    sigma_distance = (reference_makespan - avg_makespan) / std_dev if std_dev > 0 else float("inf")
    return sigma_distance

def generate_dataset(num_instances, num_jobs, num_machines, max_makespan, slack, resource_tightness):
    """
    Generate multiple scheduling instances with varying sigma distances.
    """
    dataset = []
    for _ in range(num_instances):
        duration_set_0 = np.random.randint(1, 10, num_jobs // num_machines)
        duration_set_1 = np.random.randint(1, 10, num_jobs // num_machines)

        # Generate jobs
        jobs, makespan = generate_jobs_from_duration_sets(
            num_jobs, num_machines, duration_set_0, duration_set_1, max_makespan, slack, resource_tightness
        )

        # Compute sigma distance
        sigma_distance = compute_sigma_distance(jobs, max_makespan)

        # Store the instance and its metadata
        dataset.append({"jobs": jobs, "makespan": makespan, "sigma_distance": sigma_distance})

    return dataset

# Example usage
num_instances = 10
num_jobs = 20
num_machines = 2
max_makespan = 50
slack = 0.2
resource_tightness = 5

dataset = generate_dataset(num_instances, num_jobs, num_machines, max_makespan, slack, resource_tightness)
for instance in dataset:
    print(f"Instance Sigma Distance: {instance['sigma_distance']}")

Instance Sigma Distance: -1.7528189375158045
Instance Sigma Distance: -1.7415292412933931
Instance Sigma Distance: -1.8015426229942746
Instance Sigma Distance: -1.80408518800749
Instance Sigma Distance: -1.7815461514919082
Instance Sigma Distance: -1.769011931284956
Instance Sigma Distance: -1.6241207031193157
Instance Sigma Distance: -2.002888561077362
Instance Sigma Distance: -1.893896380096151
Instance Sigma Distance: -1.8337504260933664


In [39]:
dataset

[{'jobs': {1: {'duration': 8,
    'release': 0,
    'deadline': 11,
    'machine': 0,
    'resources': 4},
   2: {'duration': 8,
    'release': 0,
    'deadline': 11,
    'machine': 0,
    'resources': 2},
   3: {'duration': 3,
    'release': 10,
    'deadline': 18,
    'machine': 0,
    'resources': 5},
   4: {'duration': 3,
    'release': 10,
    'deadline': 18,
    'machine': 0,
    'resources': 4},
   5: {'duration': 7,
    'release': 20,
    'deadline': 37,
    'machine': 0,
    'resources': 5},
   6: {'duration': 9,
    'release': 20,
    'deadline': 40,
    'machine': 0,
    'resources': 1},
   7: {'duration': 2,
    'release': 30,
    'deadline': 44,
    'machine': 0,
    'resources': 2},
   8: {'duration': 5,
    'release': 40,
    'deadline': 50,
    'machine': 0,
    'resources': 5},
   9: {'duration': 6,
    'release': 0,
    'deadline': 50,
    'machine': 0,
    'resources': 3},
   10: {'duration': 6,
    'release': 0,
    'deadline': 50,
    'machine': 0,
    'resources':